Now, let's implement an encoder-only transformer. 

# What is the difference between an encoder-only, decoder-only and encoder-decoder transformer?

From the biggest picture, we can distinguish these architectural differences like this: 

encoder-only: every token can look at the entire input sequence
decoder-only: each token can only look at earlier tokens
encoder-decoder: the encoder reads the full input, and the decoder generates the output one token at a time

We already explained that decoder-only transformers are useful for autoregressive generation. This means we want to predict next tokens from previous ones and only previous ones. We implement a casual mask for this.

For an encoder-only transformer, it will take in one sequence and produces a contextual representation of that sequence.

For example, suppose the input is: "the movie was surprisingly good." Then the representation for surprisingly can attend to every single word in the sentence. Attention is therefore bidirectional. 

# So what happens inside an encoder-only transformer?

1. we convert tokens to embeddings
2. add positional information
3. pass this information through several encoder blocks
4. each encoder block will do each self-attention over the whole sequence and then process it through a feedforward network
5. produce final hidden states

Encoder-only transformers are good for understanding tasks like what does this sentence mean?

# Example:

So an encoder-only transformer will take in the sequence "the movie was amazing" and output a sentiment "amazing"
An decoder-only transformer will take in the sequence "the movie was" and output "amazing"
An encoder-decoder transformer will take in the sequence "the movie was amazing" and output "esta película fue increíble"

In [10]:
# Again we can use a lot of similar functions from our transformer_encoder_decoder.ipynb notebook

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor

MASK_VALUE = -1e9
class TokenEmbedder(nn.Module):
    def __init__(self, vocab_size: int, d_model: int, max_len: int):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.positional_embedding = nn.Embedding(max_len, d_model)

    def forward(self, x: Tensor) -> Tensor:
        batch_size, seq_len = x.shape
        pos = torch.arange(seq_len, device=x.device).unsqueeze(0).expand(batch_size, seq_len)
        return self.token_embedding(x) + self.positional_embedding(pos)


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(d_ff, d_model)

    def forward(self, x: Tensor) -> Tensor:
        x = self.fc1(x)
        x = self.activation(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x


class LayerNorm(nn.Module):
    def __init__(self, hidden_size: int, eps: float):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(hidden_size))
        self.beta = nn.Parameter(torch.zeros(hidden_size))
        self.eps = eps

    def forward(self, x: Tensor) -> Tensor:
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        x_hat = (x - mean) / torch.sqrt(var + self.eps)
        return self.gamma * x_hat + self.beta


In [11]:
class QKVProjection(nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_k: int):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_k

        self.q_proj = nn.Linear(d_model, num_heads * d_k)
        self.k_proj = nn.Linear(d_model, num_heads * d_k)
        self.v_proj = nn.Linear(d_model, num_heads * d_k)

    def forward(self, x: Tensor) -> (Tensor, Tensor, Tensor):
        batch_size, seq_len, _ = x.shape
        q = self.q_proj(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        k = self.k_proj(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        v = self.v_proj(x).view(batch_size, seq_len, self.num_heads * self.d_k).transpose(1, 2)
        return q, k, v

In [12]:

class MultiHeadAttention(nn.Module):
    '''
    Multi-head self-attention for an encoder-only transformer.
    Each token can attend to every other token in the input sequence,
    except positions hidden by the padding mask.

    This is very similar to the multi-head attention we implemented in the transformer_decoder.ipynb notebook,
    but since this is an encoder-only architecture, the queries, keys, and values all come from the same input sequence,
    and we don't need to worry about causal masking since the encoder can attend to all positions in the input.
    '''
    def __init__(self, d_model: int, num_heads: int, d_k: int, attn_pdrop: float = 0.0):
        super().__init__()
        assert num_heads * d_k == d_model, "d_model must equal num_heads * d_k"

        self.num_heads = num_heads
        self.d_k = d_k
        self.attn_pdrop = attn_pdrop

        self.qkv_projection = QKVProjection(d_model, num_heads, d_k)
        self.out_linear = nn.Linear(d_model, d_model)

    def forward(self, x: Tensor, attention_mask: Tensor = None):
        '''
        x: input embeddings of shape (batch, seq_len, d_model)
        attention_mask: optional mask of shape (batch, seq_len)
            with 1 for real tokens and 0 for padding tokens
        '''

        # In an encoder, queries, keys, and values all come from the same input.
        Q, K, V = self.qkv_projection(x)

        # Q and K already have shape (B, H, T, d_k).
        # V comes back flattened across heads, so reshape it to match.
        B, _, T = V.shape
        V = V.contiguous().view(B, self.num_heads, self.d_k, T).transpose(-2, -1)

        # Compare each query against every key to get attention scores.
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)

        # Prevent the model from attending to padded input positions.
        if attention_mask is not None:
            mask = (attention_mask == 0).view(B, 1, 1, T)
            scores = scores.masked_fill(mask, MASK_VALUE)

        attn_weights = F.softmax(scores, dim=-1)

        if self.attn_pdrop is not None and self.training:
            attn_weights = F.dropout(attn_weights, p=self.attn_pdrop, training=True)

        # Weighted sum of values for each head.
        context = torch.matmul(attn_weights, V)

        # Merge the heads back together to recover (B, T, d_model).
        context = context.transpose(1, 2).contiguous().view(B, T, self.num_heads * self.d_k)
        output = self.out_linear(context)

        return attn_weights, output


In [13]:
# Now let's implement our Encoder Block
# Our encoder block should do two main things
# 1. Self-attention: lets each token look at all other tokens in the input sequence.
# 2. Feedforward network: applies a small MLP independently to each token position.

# structure is x -> LayerNorm -> Self-Attention -> Add & Dropout -> LayerNorm -> FeedForward -> Add & Dropout
# This is the same encoder block as the one we implemented in the transformer_encoder_decoder.ipynb notebook,
# but since this is an encoder-only architecture, we don't need to worry about cross-attention or causal masking.

class EncoderBlock(nn.Module):
    def __init__(self, d_model: int, num_heads: int, attn_pdrop: float, dropout: float, d_ff: int, eps: float):
        super().__init__()
        d_k = d_model // num_heads

        self.self_attn = MultiHeadAttention(d_model, num_heads, d_k, attn_pdrop)
        self.ff = FeedForward(d_model, d_ff, dropout)

        self.ln1 = LayerNorm(d_model, eps)
        self.ln2 = LayerNorm(d_model, eps)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: Tensor, attention_mask: Tensor = None) -> Tensor:
        # Pre-norm before self-attention.
        x_norm = self.ln1(x)

        # Encoder self-attention uses the same normalized tensor for queries, keys, and values.
        _, attn_out = self.self_attn(x_norm, attention_mask=attention_mask)

        # Residual connection around the attention block.
        x = x + self.dropout(attn_out)

        # Pre-norm before the feedforward block.
        x_norm = self.ln2(x)
        ff_out = self.ff(x_norm)

        # Residual connection around the feedforward block.
        x = x + self.dropout(ff_out)

        return x


# we can also copy over the encoder class from the transformer_encoder_decoder.ipynb notebook

class Encoder(nn.Module):
    def __init__(self, num_layers: int, d_model: int, num_heads: int, attn_pdrop: float, dropout: float, d_ff: int, eps: float):
        super().__init__()
        self.layers = nn.ModuleList()
        for _ in range(num_layers):
            self.layers.append(EncoderBlock(d_model, num_heads, attn_pdrop, dropout, d_ff, eps))

    def forward(self, x: Tensor, attention_mask: Tensor = None) -> Tensor:
        for layer in self.layers:
            x = layer(x, attention_mask=attention_mask)
        return x


In [14]:
# Now we can create our model
# This model reads a single input sequence, runs it through the encoder,
# pools the final hidden states into one vector, and predicts a class label.

class EncoderOnlyTransformer(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        num_classes: int,
        d_model: int,
        num_heads: int,
        attn_pdrop: float,
        dropout: float,
        d_ff: int,
        max_len: int,
        num_layers: int,
        eps: float,
    ):
        super().__init__()

        self.embedder = TokenEmbedder(vocab_size, d_model, max_len)
        self.encoder = Encoder(num_layers, d_model, num_heads, attn_pdrop, dropout, d_ff, eps)

        # final classifier head
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, tokens: Tensor, attention_mask: Tensor = None) -> Tensor:
        # tokens: (batch_size, seq_len)
        x = self.embedder(tokens)                          # (B, T, d_model)
        x = self.encoder(x, attention_mask=attention_mask) # (B, T, d_model)

        # Now, we need to pool the final hidden states across the sequence dimension to get a single vector for each example in the batch.

        if attention_mask is not None:
            # we first unsqueeze the attention mask to have shape (B, T, 1) so it can be broadcasted with the hidden states,
            # and convert it to float so we can do weighted sum.
            mask = attention_mask.unsqueeze(-1).float()   # (B, T, 1)
            x_sum = (x * mask).sum(dim=1)                 # (B, d_model)
            lengths = mask.sum(dim=1).clamp(min=1e-6)     # (B, 1)
            pooled = x_sum / lengths
        else:
            pooled = x.mean(dim=1)                        # (B, d_model)

        logits = self.classifier(pooled)                  # (B, num_classes)
        return logits

In [15]:
# Now we can test out our encoder with some dummy input data to make sure it runs without errors.

from torch.utils.data import Dataset, DataLoader

examples = [
    ("i love this movie", 1),
    ("this film is amazing", 1),
    ("what a great story", 1),
    ("i really enjoyed this", 1),
    ("this was wonderful", 1),
    ("i hate this movie", 0),
    ("this film is terrible", 0),
    ("what a boring story", 0),
    ("i really disliked this", 0),
    ("this was awful", 0),
]

special_tokens = ["<pad>", "<unk>", "<eos>"]

# build vocabulary
vocab_words = set()
for text, _ in examples:
    for tok in text.split():
        vocab_words.add(tok)

vocab_list = special_tokens + sorted(vocab_words)
vocab = {tok: i for i, tok in enumerate(vocab_list)}
id_to_token = {i: tok for tok, i in vocab.items()}

PAD_ID = vocab["<pad>"]
UNK_ID = vocab["<unk>"]
EOS_ID = vocab["<eos>"]

def encode_text(text):
    tokens = []
    for tok in text.split():
        if tok not in vocab:
            tokens.append(UNK_ID)
        else:
            tokens.append(vocab[tok])

    return tokens + [EOS_ID]

class ClassificationDataset(Dataset):
    def __init__(self, examples):
        self.examples = []
        for text, label in examples:
            self.examples.append((encode_text(text), label))

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]

In [16]:
def collate_classification_batch(batch):
    token_batch, label_batch = zip(*batch)

    max_len = max(len(x) for x in token_batch)

    padded_tokens = []
    attention_masks = []

    for tokens in token_batch:
        pad_len = max_len - len(tokens)
        padded = tokens + [PAD_ID] * pad_len
        mask = [1] * len(tokens) + [0] * pad_len

        padded_tokens.append(padded)
        attention_masks.append(mask)

    tokens_tensor = torch.tensor(padded_tokens, dtype=torch.long)
    attention_mask_tensor = torch.tensor(attention_masks, dtype=torch.long)
    labels_tensor = torch.tensor(label_batch, dtype=torch.long)

    return tokens_tensor, attention_mask_tensor, labels_tensor

In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dataset = ClassificationDataset(examples)
loader = DataLoader(dataset, batch_size=4, shuffle=True, collate_fn=collate_classification_batch)

model = EncoderOnlyTransformer(
    vocab_size=len(vocab),
    num_classes=2,
    d_model=64,
    num_heads=4,
    attn_pdrop=0.1,
    dropout=0.1,
    d_ff=128,
    max_len=16,
    num_layers=2,
    eps=1e-6,
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=3e-3)

num_epochs = 100

model.train()
for epoch in range(num_epochs):
    total_loss = 0.0

    for tokens, attention_mask, labels in loader:
        tokens = tokens.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        logits = model(tokens, attention_mask=attention_mask)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1}, Loss: {total_loss / len(loader):.4f}")

Epoch 10, Loss: 0.0672
Epoch 20, Loss: 0.0000
Epoch 30, Loss: 0.0000
Epoch 40, Loss: 0.0000
Epoch 50, Loss: 0.0000
Epoch 60, Loss: 0.0000
Epoch 70, Loss: 0.0000
Epoch 80, Loss: 0.0000
Epoch 90, Loss: 0.0000
Epoch 100, Loss: 0.0000


In [18]:
def predict_sentiment(model, text):
    model.eval()

    ids = encode_text(text)
    tokens = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)
    attention_mask = (tokens != PAD_ID).long()

    with torch.no_grad():
        logits = model(tokens, attention_mask=attention_mask)
        pred = logits.argmax(dim=-1).item()

    return "positive" if pred == 1 else "negative"

test_sentences = [
    "i love this",
    "this was amazing",
    "i hate this",
    "what a terrible film",
]

for sentence in test_sentences:
    prediction = predict_sentiment(model, sentence)
    print(f"{sentence} -> {prediction}")

i love this -> positive
this was amazing -> positive
i hate this -> negative
what a terrible film -> negative
